# LPBH Model

This pipeline is the LBPH-based.

It reads images from `data/face`, preprocesses faces, trains an LBPH recognizer, and saves `models/lbph.yml`.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, cast

import cv2
import numpy as np
from numpy.typing import NDArray

ImageArray = NDArray[np.uint8]

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

FACE_DIR: Path = project_root / "data" / "face"
MODELS_DIR: Path = project_root / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

LBPH_PATH: Path = MODELS_DIR / "lbph.yml"
LABEL_MAP_PATH: Path = MODELS_DIR / "label_map_lbph.txt"

OUTPUT_SIZE: tuple[int, int] = (128, 128)
CLAHE_CLIP_LIMIT: float = 2.0
CLAHE_GRID_SIZE: tuple[int, int] = (8, 8)
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

face_module: Any = getattr(cv2, "face", None)
if face_module is None or getattr(face_module, "LBPHFaceRecognizer_create", None) is None:
    raise RuntimeError("LBPH is unavailable. Install opencv-contrib-python in your environment.")

lbph_factory = cast(Any, face_module.LBPHFaceRecognizer_create)

print(f"Face folder: {FACE_DIR}")
print(f"LBPH output: {LBPH_PATH}")
print(f"Label map : {LABEL_MAP_PATH}")

Face folder: c:\Users\harry\Documents\school\ip\AttSystem\data\face
LBPH output: c:\Users\harry\Documents\school\ip\AttSystem\models\lbph.yml
Label map : c:\Users\harry\Documents\school\ip\AttSystem\models\label_map_lbph.txt


In [2]:
haar_root = Path(getattr(cv2, "data").haarcascades)  # type: ignore[attr-defined]
face_cascade = cv2.CascadeClassifier(str(haar_root / "haarcascade_frontalface_default.xml"))
if face_cascade.empty():
    raise RuntimeError("Failed to load OpenCV frontal face cascade.")

clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_GRID_SIZE)


def detect_face_roi(gray: ImageArray) -> ImageArray:
    h, w = gray.shape[:2]
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40),
    )

    if len(faces) == 0:
        side: int = int(min(h, w) * 0.75)
        cx, cy = w // 2, h // 2
        x1: int = max(0, cx - side // 2)
        y1: int = max(0, cy - side // 2)
        x2: int = min(w, x1 + side)
        y2: int = min(h, y1 + side)
        return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)

    x, y, fw, fh = max(faces, key=lambda f: int(f[2]) * int(f[3]))
    pad_w: int = int(fw * 0.2)
    pad_h: int = int(fh * 0.2)

    x1: int = max(0, int(x) - pad_w)
    y1: int = max(0, int(y) - pad_h)
    x2: int = min(w, int(x) + int(fw) + pad_w)
    y2: int = min(h, int(y) + int(fh) + pad_h)

    return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)


def preprocess_roi(roi_gray: ImageArray) -> ImageArray:
    enhanced = clahe.apply(roi_gray)
    enhanced = cv2.GaussianBlur(enhanced, (3, 3), 0)
    resized = cv2.resize(enhanced, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
    return resized.astype(np.uint8, copy=False)


def build_lbph_dataset(face_dir: Path) -> tuple[list[ImageArray], NDArray[np.int32], list[str]]:
    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError("No person folders found in data/face.")

    X: list[ImageArray] = []
    y: list[int] = []
    label_names: list[str] = []

    total_images: int = 0
    kept_images: int = 0

    for label_id, person_dir in enumerate(person_dirs):
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        label_names.append(person_dir.name)
        person_kept: int = 0

        for f in files:
            total_images += 1
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue

            gray = cv2.cvtColor(bgr.astype(np.uint8, copy=False), cv2.COLOR_BGR2GRAY).astype(np.uint8, copy=False)
            roi = detect_face_roi(gray)
            if roi.size == 0:
                continue

            sample = preprocess_roi(roi)
            X.append(sample)
            y.append(label_id)
            person_kept += 1
            kept_images += 1

        print(f"{person_dir.name}: used {person_kept}/{len(files)} images")

    if len(X) == 0:
        raise RuntimeError("No usable samples built from dataset.")

    print(f"Total read: {total_images}, usable: {kept_images}")
    return X, np.array(y, dtype=np.int32), label_names

In [3]:
def train_and_save_lbph() -> None:
    X, y, label_names = build_lbph_dataset(FACE_DIR)

    model = lbph_factory()
    model.setRadius(1)
    model.setNeighbors(8)
    model.setGridX(8)
    model.setGridY(8)
    model.train(X, y)
    model.save(str(LBPH_PATH))

    with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        for idx, name in enumerate(label_names):
            f.write(f"{idx},{name}\n")

    print("\nPipeline 2 (LBPH) complete")
    print(f"Model saved: {LBPH_PATH}")
    print(f"Label map : {LABEL_MAP_PATH}")
    print(f"Samples   : {len(X)}")
    print(f"People    : {len(label_names)}")


train_and_save_lbph()

benjamin: used 1080/1080 images
chern_tak: used 1008/1008 images
chillien: used 324/324 images
daniel: used 1080/1080 images
dylan: used 756/756 images
han_soon: used 720/720 images
harry: used 360/360 images
isaac: used 360/360 images
jing_ang: used 1080/1080 images
jun_wei: used 972/972 images
kang_kai: used 1080/1080 images
marion: used 1008/1008 images
ms_nurul: used 396/396 images
qi_xuan: used 1080/1080 images
shuang_quan: used 1620/1620 images
wee_xuan: used 864/864 images
xiang_yue: used 1080/1080 images
xu_sheng: used 1044/1044 images
yoke_hong: used 1152/1152 images
yong_kang: used 864/864 images
zi_herng: used 360/360 images
Total read: 18288, usable: 18288

Pipeline 2 (LBPH) complete
Model saved: c:\Users\harry\Documents\school\ip\AttSystem\models\lbph.yml
Label map : c:\Users\harry\Documents\school\ip\AttSystem\models\label_map_lbph.txt
Samples   : 18288
People    : 21
